[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/03_sql-foundations.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 3 of 21*

# SQL Foundations

The most important skill in data engineering, from your very first query: SELECT, WHERE, sorting, aggregates, GROUP BY, HAVING, all the joins, NULLs, CASE and subqueries, on a real database.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Read SQL in the order it really runs: gather, filter, group, filter groups, pick, sort. Most confusion disappears.

**What interviewers really test here:** Joins and row counts, NULL traps, WHERE vs HAVING. Expect SQL in almost every data interview.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Your first queries *(Basics)*
- [ ] 2. Summarising: GROUP BY and HAVING *(Basics)*
- [ ] 3. Joins, and why totals go wrong *(Core)*
- [ ] 4. Subqueries, EXISTS and UNION *(Core)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### NULL rules

- `x = NULL` is never true; use `IS NULL`.
- `COUNT(*)` counts rows, and `COUNT(col)` skips NULLs.
- SUM and AVG ignore NULLs, so an average divides only by non-NULL rows.
- `NOT IN` with a NULL in the list returns nothing.
- In GROUP BY, NULLs form one group.

---
# Part 2: Lessons, step by step

## Lesson 1: Your first queries

*Level: Basics*

A database table is a spreadsheet with rules: each column has a type, and every row has the same columns. **SQL is how you ask it questions.** It's the most important skill in this course, and you'll use it daily for the rest of your career.



You describe *what* you want, and the database works out *how*:



```python
SELECT name, price       -- which columns
FROM products            -- from which table
WHERE price > 1          -- which rows
ORDER BY price           -- in what order
LIMIT 10;                -- how many
```



| Filter | Example |
| --- | --- |
| Comparison | `price > 1`, `status = 'paid'` |
| Combine | `a AND b`, `a OR b` |
| A list | `product_id IN (10, 12)` |
| A range | `day BETWEEN '2026-01-06' AND '2026-02-01'` (inclusive) |
| Pattern | `name LIKE 'A%'` means "starts with A" |



**NULL means unknown**, and it's not zero or empty. Anything compared to unknown is unknown, so `city = NULL` finds nothing. Use `IS NULL`. For a default, use `COALESCE(city, 'unknown')`.



**Practise as you read:** the [SQL IDE](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/ide/sql.html) runs real SQL in your browser, with sample databases and questions that check your answer. Keep it open in another tab, and try each example.



**Good habits from day one:** name your columns rather than using `SELECT *`, because tables change and scans cost money. Format queries over several lines, as you'll read them more often than you write them.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q4](#q4), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 2: Summarising: GROUP BY and HAVING

*Level: Basics*

**Aggregates** squeeze many rows into one number: COUNT, SUM, AVG, MIN, MAX. Prices 2, 100, 5, 20 give a count of 4, a minimum of 2, a maximum of 100 and an average of 31.75.



**GROUP BY** makes one result per group. Picture sorting receipts into piles, then totalling each pile.



**The order SQL actually runs in explains nearly every beginner error:**



| Order | Clause | Does |
| --- | --- | --- |
| 1 | FROM, JOIN | gather rows |
| 2 | WHERE | keep rows |
| 3 | GROUP BY | make piles |
| 4 | HAVING | keep piles |
| 5 | SELECT | pick and compute columns |
| 6 | ORDER BY | sort |
| 7 | LIMIT | cut off |



So `WHERE COUNT(*) > 1` fails, because counts don't exist yet at step 2. That's what `HAVING` is for.



**COUNT trap:** with 7 orders where one status is NULL, `COUNT(*)` is 7, `COUNT(status)` is 6, and `COUNT(DISTINCT status)` is 2.



**Conditional aggregation** is very useful, as it counts several things in one pass:



```python
SELECT user_id,
       COUNT(*) AS total,
       SUM(CASE WHEN status = 'paid' THEN 1 ELSE 0 END) AS paid
FROM orders GROUP BY user_id;
```



User 2 has 2 orders, with 1 paid, so 50%.

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q6](#q6), [Q7](#q7), [Q9](#q9), [Q11](#q11), [Q18](#q18), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 3: Joins, and why totals go wrong

*Level: Core*

Good databases store each fact once, in separate tables, so you join them to answer questions. Our demo has orders for Asha and Ben, plus users Asha, Ben and Eli.



| Join | Keeps | Eli (no orders)? |
| --- | --- | --- |
| INNER | only matching rows | gone |
| LEFT (users first) | every user | kept, with NULL for order columns |
| FULL | all rows from both | kept |
| CROSS | every combination | n × m rows |



Run the demo and see for yourself.



**Finding what's missing** is a classic: LEFT JOIN, then keep rows where the right side is NULL. Why not `NOT IN`? Because `x NOT IN (1, NULL)` is never true, so one NULL makes it return nothing.



**The most expensive beginner mistake: joins multiply rows.** If a user has 2 rows on the right, each of their orders appears twice. In our practice data, the true total quantity is 20, but joining to a table with 2 rows per user gives **40**. Nothing errors, and the report's simply wrong by double.



**Always ask: what's one row of each table?** And check counts before and after a join. A common fix is to aggregate first, then join, or to deduplicate the other side.



**Filtering a LEFT JOIN trap:** a `WHERE` on the right table removes the NULL rows, silently turning it into an INNER JOIN. Move that condition into `ON`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import sqlite3
DEMO = """
CREATE TABLE orders (id INT, user TEXT, amount INT);
INSERT INTO orders VALUES (1,'Asha',10), (2,'Asha',20), (3,'Ben',5);
CREATE TABLE users (name TEXT, city TEXT);
INSERT INTO users VALUES ('Asha','Pune'), ('Ben','Delhi'), ('Eli','Goa');
"""
def sql(query):
    con = sqlite3.connect(":memory:")
    con.executescript(DEMO)
    return con.execute(query).fetchall()


def joins_side_by_side():
    print("INNER:", sql("SELECT u.name, o.amount FROM users u JOIN orders o ON o.user = u.name ORDER BY 1, 2"))
    print("LEFT: ", sql("SELECT u.name, o.amount FROM users u LEFT JOIN orders o ON o.user = u.name ORDER BY 1, 2"))
    print("Missing:", sql("SELECT u.name FROM users u LEFT JOIN orders o ON o.user = u.name WHERE o.id IS NULL"))


joins_side_by_side()

INNER: [('Asha', 10), ('Asha', 20), ('Ben', 5)]
LEFT:  [('Asha', 10), ('Asha', 20), ('Ben', 5), ('Eli', None)]
Missing: [('Eli',)]


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q13](#q13), [Q14](#q14), [Q15](#q15), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 4: Subqueries, EXISTS and UNION

*Level: Core*

A **subquery** is a query inside a query:



```python
SELECT name FROM products
WHERE price > (SELECT AVG(price) FROM products);
```



The average of 2, 100, 5, 20 is 31.75, so only Desk qualifies.



**EXISTS** asks "is there at least one?", and stops at the first match. It's often the clearest way of writing "users who have a refunded order", and it can't accidentally multiply rows the way a join can.



**UNION** stacks results, and removes duplicates. **UNION ALL** keeps everything, and it's faster because there's no duplicate check. Use UNION ALL unless you need deduplication. Cities Pune, Delhi, Pune, NULL, Goa stacked twice give 10 rows with UNION ALL, and 4 with UNION.



**Dates.** Group by month with `substr(day, 1, 7)` in SQLite, or `DATE_TRUNC('month', day)` in PostgreSQL and most warehouses.



**Next:** window functions, which is where SQL starts being really powerful.

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q17](#q17), [Q12](#q12). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
import sqlite3

SCHEMA = """
CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup TEXT);
INSERT INTO users VALUES (1,'Asha','Pune','2026-01-01'), (2,'Ben','Delhi','2026-01-03'), (3,'Chen','Pune','2026-02-10'),
 (4,'Dara',NULL,'2026-02-11'), (5,'Eli','Goa','2026-03-01');
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL);
INSERT INTO products VALUES (10,'Pen','office',2), (11,'Desk','office',100), (12,'Tea','food',5), (13,'Lamp','home',20);
CREATE TABLE orders (id INTEGER PRIMARY KEY, user_id INTEGER, product_id INTEGER, qty INTEGER, day TEXT, status TEXT);
INSERT INTO orders VALUES (1,1,10,10,'2026-01-05','paid'), (2,1,11,1,'2026-01-06','paid'), (3,2,12,4,'2026-01-06','paid'),
 (4,2,10,1,'2026-02-01','refunded'), (5,3,12,2,'2026-02-15','paid'), (6,1,12,1,'2026-02-20','paid'), (7,4,11,1,'2026-03-02',NULL);
"""

def run_sql(query):
    """Run a query on a fresh database and return the rows as lists."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    rows = con.execute(query).fetchall()
    con.close()
    return [list(r) for r in rows]


<a id="q1"></a>
### Q1. Select Some Columns

🟢 Warm-up · *SELECT, ORDER BY*

Return the name and city of every user, ordered by name.

**Examples**

- `` → `[['Asha', 'Pune'], ['Ben', 'Delhi'], ['Chen', 'Pune'], ['Dara', None], ['Eli', 'Goa']]`

In [ ]:
def names_cities():
    """
    Return the name and city of every user, ordered by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(names_cities, [
    ((), [['Asha', 'Pune'], ['Ben', 'Delhi'], ['Chen', 'Pune'], ['Dara', None], ['Eli', 'Goa']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch everything and pick columns in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Name the columns, and let the database sort.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch everything and pick columns in Python.

```python
def names_cities():
    rows = run_sql("SELECT * FROM users")
    return sorted(([r[1], r[2]] for r in rows), key=lambda r: r[0])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Name the columns, and let the database sort.

```python
def names_cities():
    return run_sql("SELECT name, city FROM users ORDER BY name")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch everything and pick columns in Python. That costs O(n log n) time and O(n) space."

"I can do better: Name the columns, and let the database sort. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/select-some-columns)

<a id="q2"></a>
### Q2. Filter Rows

🟢 Warm-up · *WHERE*

Return names of products in the office category costing more than 1, cheapest first.

**Examples**

- `` → `['Pen', 'Desk']`

In [ ]:
def office_products():
    """
    Return names of products in the office category costing more than 1,
    cheapest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(office_products, [
    ((), ['Pen', 'Desk']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A nested filter through a subquery.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`WHERE` with `AND`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A nested filter through a subquery.

```python
def office_products():
    rows = run_sql("SELECT name FROM (SELECT * FROM products WHERE category = 'office') WHERE price > 1 ORDER BY price")
    return [r[0] for r in rows]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `WHERE` with `AND`.

```python
def office_products():
    return [r[0] for r in run_sql("SELECT name FROM products WHERE category = 'office' AND price > 1 ORDER BY price")]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A nested filter through a subquery. That costs O(n log n) time and O(n) space."

"I can do better: `WHERE` with `AND`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/filter-rows)

<a id="q3"></a>
### Q3. IN, BETWEEN and LIKE

🟢 Warm-up · *More filters*

Return ids of orders on days from 2026-01-06 to 2026-02-01 inclusive, whose product\_id is 10 or 12. Order by id.

**Examples**

- `` → `[3, 4]`

In [ ]:
def filtered_orders():
    """
    Return ids of orders on days from 2026-01-06 to 2026-02-01 inclusive, whose
    product_id is 10 or 12. Order by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(filtered_orders, [
    ((), [3, 4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Spell out every comparison.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`BETWEEN` and `IN` read better.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Spell out every comparison.

```python
def filtered_orders():
    return [r[0] for r in run_sql("""SELECT id FROM orders
        WHERE day >= '2026-01-06' AND day <= '2026-02-01' AND (product_id = 10 OR product_id = 12) ORDER BY id""")]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `BETWEEN` and `IN` read better.

```python
def filtered_orders():
    return [r[0] for r in run_sql("""SELECT id FROM orders
        WHERE day BETWEEN '2026-01-06' AND '2026-02-01' AND product_id IN (10, 12) ORDER BY id""")]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Spell out every comparison. That costs O(n) time and O(n) space."

"I can do better: `BETWEEN` and `IN` read better. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/in-between-and-like)

<a id="q4"></a>
### Q4. Finding NULLs

🟢 Warm-up · *NULL isn't a value*

Return names of users with no city. `city = NULL` never matches anything.

**Examples**

- `` → `['Dara']`

In [ ]:
def no_city():
    """
    Return names of users with no city. city = NULL never matches anything.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(no_city, [
    ((), ['Dara']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch all and check for None in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`IS NULL`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch all and check for None in Python.

```python
def no_city():
    return [r[0] for r in run_sql("SELECT name, city FROM users") if r[1] is None]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `IS NULL`.

```python
def no_city():
    return [r[0] for r in run_sql("SELECT name FROM users WHERE city IS NULL")]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch all and check for None in Python. That costs O(n) time and O(n) space."

"I can do better: `IS NULL`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/finding-nulls)

<a id="q5"></a>
### Q5. Aggregates

🟢 Warm-up · *COUNT, SUM, AVG, MIN, MAX*

Return the product count, cheapest price, highest price and average price rounded to 2 decimals, as one row.

**Examples**

- `` → `[4, 2.0, 100.0, 31.75]`

In [ ]:
def price_stats():
    """
    Return the product count, cheapest price, highest price and average price
    rounded to 2 decimals, as one row.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(price_stats, [
    ((), [4, 2.0, 100.0, 31.75]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch prices and compute in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Aggregates in one query.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch prices and compute in Python.

```python
def price_stats():
    p = [r[0] for r in run_sql("SELECT price FROM products")]
    return [len(p), min(p), max(p), round(sum(p) / len(p), 2)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Aggregates in one query.

```python
def price_stats():
    return run_sql("SELECT COUNT(*), MIN(price), MAX(price), ROUND(AVG(price), 2) FROM products")[0]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch prices and compute in Python. That costs O(n) time and O(n) space."

"I can do better: Aggregates in one query. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/aggregates)

<a id="q6"></a>
### Q6. GROUP BY

🟢 Warm-up · *One row per group*

Return each category and its product count, largest count first, then by category.

**Examples**

- `` → `[['office', 2], ['food', 1], ['home', 1]]`

In [ ]:
def category_counts():
    """
    Return each category and its product count, largest count first, then by
    category.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(category_counts, [
    ((), [['office', 2], ['food', 1], ['home', 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`GROUP BY`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Count in Python.

```python
def category_counts():
    counts = {}
    for r in run_sql("SELECT category FROM products"):
        counts[r[0]] = counts.get(r[0], 0) + 1
    return sorted(([k, v] for k, v in counts.items()), key=lambda x: (-x[1], x[0]))
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `GROUP BY`.

```python
def category_counts():
    return run_sql("SELECT category, COUNT(*) AS n FROM products GROUP BY category ORDER BY n DESC, category")
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count in Python. That costs O(n) time and O(c) space."

"I can do better: `GROUP BY`. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/group-by)

<a id="q7"></a>
### Q7. HAVING

🟢 Warm-up · *Filter groups*

Return user\_ids with 2 or more orders, with their count, ordered by user\_id.

**Examples**

- `` → `[[1, 3], [2, 2]]`

In [ ]:
def repeat_buyers():
    """
    Return user_ids with 2 or more orders, with their count, ordered by user_id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(repeat_buyers, [
    ((), [[1, 3], [2, 2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Group in a subquery, then use WHERE outside.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`HAVING`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Group in a subquery, then use WHERE outside.

```python
def repeat_buyers():
    return run_sql("SELECT user_id, n FROM (SELECT user_id, COUNT(*) AS n FROM orders GROUP BY user_id) WHERE n >= 2 ORDER BY user_id")
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `HAVING`.

```python
def repeat_buyers():
    return run_sql("SELECT user_id, COUNT(*) FROM orders GROUP BY user_id HAVING COUNT(*) >= 2 ORDER BY user_id")
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Group in a subquery, then use WHERE outside. That costs O(n) time and O(u) space."

"I can do better: `HAVING`. That's O(n) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/having)

<a id="q8"></a>
### Q8. INNER JOIN

🟢 Warm-up · *Combine tables*

Return order id, user name and product name for each order, ordered by order id.

**Examples**

- `` → `[[1, 'Asha', 'Pen'], [2, 'Asha', 'Desk'], [3, 'Ben', 'Tea'], [4, 'Ben', 'Pen'], [5, 'Chen', 'Tea'], [6, 'Asha', 'Tea'], [7, 'Dara', 'Desk']]`

In [ ]:
def order_details():
    """
    Return order id, user name and product name for each order, ordered by order
    id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(order_details, [
    ((), [[1, 'Asha', 'Pen'], [2, 'Asha', 'Desk'], [3, 'Ben', 'Tea'], [4, 'Ben', 'Pen'], [5, 'Chen', 'Tea'], [6, 'Asha', 'Tea'], [7, 'Dara', 'Desk']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scalar subqueries for each name.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two joins.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Scalar subqueries for each name.

```python
def order_details():
    return run_sql("""SELECT id, (SELECT name FROM users u WHERE u.id = o.user_id),
        (SELECT name FROM products p WHERE p.id = o.product_id) FROM orders o ORDER BY id""")
```

**Cost:** O(n·m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two joins.

```python
def order_details():
    return run_sql("""SELECT o.id, u.name, p.name FROM orders o
        JOIN users u ON u.id = o.user_id JOIN products p ON p.id = o.product_id ORDER BY o.id""")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scalar subqueries for each name. That costs O(n·m) time and O(n) space."

"I can do better: Two joins. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/inner-join)

<a id="q9"></a>
### Q9. CASE Expressions

🟢 Warm-up · *If/else in SQL*

Label each product 'cheap' (below 10) or 'premium', and return the count per label, ordered by label.

**Examples**

- `` → `[['cheap', 2], ['premium', 2]]`

In [ ]:
def price_bands():
    """
    Label each product 'cheap' (below 10) or 'premium', and return the count per
    label, ordered by label.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(price_bands, [
    ((), [['cheap', 2], ['premium', 2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two counting queries.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Group by a CASE.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Two counting queries.

```python
def price_bands():
    cheap = run_sql("SELECT COUNT(*) FROM products WHERE price < 10")[0][0]
    prem = run_sql("SELECT COUNT(*) FROM products WHERE price >= 10")[0][0]
    return [["cheap", cheap], ["premium", prem]]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Group by a CASE.

```python
def price_bands():
    return run_sql("""SELECT CASE WHEN price < 10 THEN 'cheap' ELSE 'premium' END AS band, COUNT(*)
        FROM products GROUP BY band ORDER BY band""")
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two counting queries. That costs O(n) time and O(1) space."

"I can do better: Group by a CASE. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/case-expressions)

<a id="q10"></a>
### Q10. COALESCE

🟢 Warm-up · *Defaults for NULL*

Return each user's name with their city, or 'unknown' when NULL, ordered by id.

**Examples**

- `` → `[['Asha', 'Pune'], ['Ben', 'Delhi'], ['Chen', 'Pune'], ['Dara', 'unknown'], ['Eli', 'Goa']]`

In [ ]:
def city_or_unknown():
    """
    Return each user's name with their city, or 'unknown' when NULL, ordered by
    id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(city_or_unknown, [
    ((), [['Asha', 'Pune'], ['Ben', 'Delhi'], ['Chen', 'Pune'], ['Dara', 'unknown'], ['Eli', 'Goa']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Replace in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`COALESCE(city, 'unknown')`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Replace in Python.

```python
def city_or_unknown():
    return [[r[0], r[1] if r[1] is not None else "unknown"] for r in run_sql("SELECT name, city FROM users ORDER BY id")]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `COALESCE(city, 'unknown')`.

```python
def city_or_unknown():
    return run_sql("SELECT name, COALESCE(city, 'unknown') FROM users ORDER BY id")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Replace in Python. That costs O(n) time and O(n) space."

"I can do better: `COALESCE(city, 'unknown')`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/coalesce)

<a id="q11"></a>
### Q11. COUNT Traps

🟢 Warm-up · *Rows vs values*

Return COUNT(\*), COUNT(status) and COUNT(DISTINCT status) from orders, as one row.

**Examples**

- `` → `[7, 6, 2]`

In [ ]:
def count_traps():
    """
    Return COUNT(*), COUNT(status) and COUNT(DISTINCT status) from orders, as
    one row.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_traps, [
    ((), [7, 6, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute from fetched values.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One query. Only `COUNT(*)` counts NULLs.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute from fetched values.

```python
def count_traps():
    v = [r[0] for r in run_sql("SELECT status FROM orders")]
    known = [x for x in v if x is not None]
    return [len(v), len(known), len(set(known))]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One query. Only `COUNT(*)` counts NULLs.

```python
def count_traps():
    return run_sql("SELECT COUNT(*), COUNT(status), COUNT(DISTINCT status) FROM orders")[0]
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute from fetched values. That costs O(n) time and O(n) space."

"I can do better: One query. Only `COUNT(*)` counts NULLs. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/count-traps)

<a id="q12"></a>
### Q12. UNION vs UNION ALL

🟢 Warm-up · *Stacking rows*

Return row counts for stacking user cities with itself, using UNION and UNION ALL, as (union, union\_all).

**Examples**

- `` → `(4, 10)`

In [ ]:
def union_counts():
    """
    Return row counts for stacking user cities with itself, using UNION and
    UNION ALL, as (union, union_all).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(union_counts, [
    ((), (4, 10)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two separate queries.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Both counts in one query. UNION removes duplicates, including NULL treated as one.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Two separate queries.

```python
def union_counts():
    u = run_sql("SELECT city FROM users UNION SELECT city FROM users")
    a = run_sql("SELECT city FROM users UNION ALL SELECT city FROM users")
    return (len(u), len(a))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Both counts in one query. UNION removes duplicates, including NULL treated as one.

```python
def union_counts():
    r = run_sql("""SELECT (SELECT COUNT(*) FROM (SELECT city FROM users UNION SELECT city FROM users)),
                          (SELECT COUNT(*) FROM (SELECT city FROM users UNION ALL SELECT city FROM users))""")[0]
    return (r[0], r[1])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two separate queries. That costs O(n) time and O(n) space."

"I can do better: Both counts in one query. UNION removes duplicates, including NULL treated as one. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/union-vs-union-all)

<a id="q13"></a>
### Q13. LEFT JOIN to Find What's Missing

🟡 Interview standard · *Users with no orders*

Return names of users who have never ordered, and names of products never ordered, as a tuple of two lists.

**Examples**

- `` → `(['Eli'], ['Lamp'])`

In [ ]:
def never_ordered():
    """
    Return names of users who have never ordered, and names of products never
    ordered, as a tuple of two lists.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(never_ordered, [
    ((), (['Eli'], ['Lamp'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

`NOT IN`. Beware of NULLs in the subquery.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`LEFT JOIN ... WHERE right.id IS NULL`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** `NOT IN`. Beware of NULLs in the subquery.

```python
def never_ordered():
    u = run_sql("SELECT name FROM users WHERE id NOT IN (SELECT user_id FROM orders)")
    p = run_sql("SELECT name FROM products WHERE id NOT IN (SELECT product_id FROM orders)")
    return ([r[0] for r in u], [r[0] for r in p])
```

**Cost:** O(n·m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `LEFT JOIN ... WHERE right.id IS NULL`.

```python
def never_ordered():
    u = run_sql("SELECT u.name FROM users u LEFT JOIN orders o ON o.user_id = u.id WHERE o.id IS NULL")
    p = run_sql("SELECT p.name FROM products p LEFT JOIN orders o ON o.product_id = p.id WHERE o.id IS NULL")
    return ([r[0] for r in u], [r[0] for r in p])
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: `NOT IN`. Beware of NULLs in the subquery. That costs O(n·m) time and O(n) space."

"I can do better: `LEFT JOIN ... WHERE right.id IS NULL`. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/left-join-to-find-whats-missing)

<a id="q14"></a>
### Q14. Revenue per Category

🟡 Interview standard · *Join, then aggregate*

Return category and revenue (qty × price) from paid orders, highest first.

**Examples**

- `` → `[['office', 120.0], ['food', 35.0]]`

In [ ]:
def category_revenue():
    """
    Return category and revenue (qty × price) from paid orders, highest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(category_revenue, [
    ((), [['office', 120.0], ['food', 35.0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch joined rows and add up in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Join, filter, group.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch joined rows and add up in Python.

```python
def category_revenue():
    totals = {}
    for cat, value in run_sql("""SELECT p.category, o.qty * p.price FROM orders o
            JOIN products p ON p.id = o.product_id WHERE o.status = 'paid'"""):
        totals[cat] = totals.get(cat, 0) + value
    return sorted(([k, v] for k, v in totals.items()), key=lambda x: -x[1])
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Join, filter, group.

```python
def category_revenue():
    return run_sql("""SELECT p.category, SUM(o.qty * p.price) AS rev FROM orders o
        JOIN products p ON p.id = o.product_id WHERE o.status = 'paid'
        GROUP BY p.category ORDER BY rev DESC""")
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch joined rows and add up in Python. That costs O(n) time and O(c) space."

"I can do better: Join, filter, group. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/revenue-per-category)

<a id="q15"></a>
### Q15. Joins Multiply Rows

🟡 Interview standard · *The classic revenue bug*

Return (total qty from orders alone, total qty after wrongly joining to a table with two rows per user). Use a CTE as the second table.

**Examples**

- `` → `(20, 40)`

In [ ]:
def join_fanout():
    """
    Return (total qty from orders alone, total qty after wrongly joining to a
    table with two rows per user). Use a CTE as the second table.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(join_fanout, [
    ((), (20, 40)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Run each query separately.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A single query with a subquery for the correct value shows it side by side.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Run each query separately.

```python
def join_fanout():
    right = run_sql("SELECT SUM(qty) FROM orders")[0][0]
    wrong = run_sql("""WITH tags(user_id, tag) AS (SELECT id, 'a' FROM users UNION ALL SELECT id, 'b' FROM users)
        SELECT SUM(o.qty) FROM orders o JOIN tags t ON t.user_id = o.user_id""")[0][0]
    return (right, wrong)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A single query with a subquery for the correct value shows it side by side.

```python
def join_fanout():
    row = run_sql("""WITH tags(user_id, tag) AS (SELECT id, 'a' FROM users UNION ALL SELECT id, 'b' FROM users)
        SELECT (SELECT SUM(qty) FROM orders), SUM(o.qty) FROM orders o JOIN tags t ON t.user_id = o.user_id""")[0]
    return (row[0], row[1])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Run each query separately. That costs O(n) time and O(1) space."

"I can do better: A single query with a subquery for the correct value shows it side by side. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/joins-multiply-rows)

<a id="q16"></a>
### Q16. Subquery vs Average

🟡 Interview standard · *Compare against a total*

Return names of products priced above the average price, ordered by name.

**Examples**

- `` → `['Desk']`

In [ ]:
def above_average():
    """
    Return names of products priced above the average price, ordered by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(above_average, [
    ((), ['Desk']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the average first, then filter in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A subquery in WHERE.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Compute the average first, then filter in Python.

```python
def above_average():
    rows = run_sql("SELECT name, price FROM products")
    avg = sum(r[1] for r in rows) / len(rows)
    return sorted(r[0] for r in rows if r[1] > avg)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A subquery in WHERE.

```python
def above_average():
    return [r[0] for r in run_sql("SELECT name FROM products WHERE price > (SELECT AVG(price) FROM products) ORDER BY name")]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the average first, then filter in Python. That costs O(n) time and O(n) space."

"I can do better: A subquery in WHERE. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/subquery-vs-average)

<a id="q17"></a>
### Q17. EXISTS

🟡 Interview standard · *Is there any?*

Return names of users who have at least one refunded order.

**Examples**

- `` → `['Ben']`

In [ ]:
def refunded_users():
    """
    Return names of users who have at least one refunded order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(refunded_users, [
    ((), ['Ben']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Join with DISTINCT.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`EXISTS` stops at the first match.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Join with DISTINCT.

```python
def refunded_users():
    return [r[0] for r in run_sql("""SELECT DISTINCT u.name FROM users u JOIN orders o ON o.user_id = u.id
        WHERE o.status = 'refunded'""")]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `EXISTS` stops at the first match.

```python
def refunded_users():
    return [r[0] for r in run_sql("""SELECT name FROM users u
        WHERE EXISTS (SELECT 1 FROM orders o WHERE o.user_id = u.id AND o.status = 'refunded')""")]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Join with DISTINCT. That costs O(n) time and O(n) space."

"I can do better: `EXISTS` stops at the first match. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/exists)

<a id="q18"></a>
### Q18. Monthly Totals

🟡 Interview standard · *Grouping by date*

Return month (YYYY-MM) and number of paid orders, in month order.

**Examples**

- `` → `[['2026-01', 3], ['2026-02', 2]]`

In [ ]:
def monthly_orders():
    """
    Return month (YYYY-MM) and number of paid orders, in month order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(monthly_orders, [
    ((), [['2026-01', 3], ['2026-02', 2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Slice the date in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`substr(day, 1, 7)`. Real databases have `DATE_TRUNC`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Slice the date in Python.

```python
def monthly_orders():
    counts = {}
    for r in run_sql("SELECT day FROM orders WHERE status = 'paid'"):
        counts[r[0][:7]] = counts.get(r[0][:7], 0) + 1
    return [[k, counts[k]] for k in sorted(counts)]
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `substr(day, 1, 7)`. Real databases have `DATE_TRUNC`.

```python
def monthly_orders():
    return run_sql("SELECT substr(day, 1, 7) AS m, COUNT(*) FROM orders WHERE status = 'paid' GROUP BY m ORDER BY m")
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Slice the date in Python. That costs O(n) time and O(m) space."

"I can do better: `substr(day, 1, 7)`. Real databases have `DATE_TRUNC`. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/monthly-totals)

<a id="q19"></a>
### Q19. Top Spender

🟡 Interview standard · *Putting it together*

Return the name and total paid spend of the top-spending user.

**Examples**

- `` → `['Asha', 125.0]`

In [ ]:
def top_spender():
    """
    Return the name and total paid spend of the top-spending user.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_spender, [
    ((), ['Asha', 125.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum per user, then find the maximum in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Join, group, order, limit.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Sum per user, then find the maximum in Python.

```python
def top_spender():
    rows = run_sql("""SELECT u.name, o.qty * p.price FROM orders o JOIN users u ON u.id = o.user_id
        JOIN products p ON p.id = o.product_id WHERE o.status = 'paid'""")
    totals = {}
    for name, value in rows:
        totals[name] = totals.get(name, 0) + value
    best = max(totals, key=totals.get)
    return [best, totals[best]]
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Join, group, order, limit.

```python
def top_spender():
    return run_sql("""SELECT u.name, SUM(o.qty * p.price) AS s FROM orders o
        JOIN users u ON u.id = o.user_id JOIN products p ON p.id = o.product_id
        WHERE o.status = 'paid' GROUP BY u.id ORDER BY s DESC LIMIT 1""")[0]
```

**Cost:** O(n log u) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum per user, then find the maximum in Python. That costs O(n) time and O(u) space."

"I can do better: Join, group, order, limit. That's O(n log u) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/top-spender)

<a id="q20"></a>
### Q20. Share of Paid Orders

🟡 Interview standard · *Conditional aggregation*

Return per user\_id: total orders, paid orders and the paid percentage rounded to 1 decimal, ordered by user\_id.

**Examples**

- `` → `[[1, 3, 3, 100.0], [2, 2, 1, 50.0], [3, 1, 1, 100.0], [4, 1, 0, 0.0]]`

In [ ]:
def paid_share():
    """
    Return per user_id: total orders, paid orders and the paid percentage
    rounded to 1 decimal, ordered by user_id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(paid_share, [
    ((), [[1, 3, 3, 100.0], [2, 2, 1, 50.0], [3, 1, 1, 100.0], [4, 1, 0, 0.0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two grouped queries, joined.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`SUM(CASE ...)`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Two grouped queries, joined.

```python
def paid_share():
    total = dict(map(tuple, run_sql("SELECT user_id, COUNT(*) FROM orders GROUP BY user_id")))
    paid = dict(map(tuple, run_sql("SELECT user_id, COUNT(*) FROM orders WHERE status = 'paid' GROUP BY user_id")))
    return [[u, total[u], paid.get(u, 0), round(100 * paid.get(u, 0) / total[u], 1)] for u in sorted(total)]
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `SUM(CASE ...)`.

```python
def paid_share():
    return run_sql("""SELECT user_id, COUNT(*), SUM(CASE WHEN status = 'paid' THEN 1 ELSE 0 END),
        ROUND(100.0 * SUM(CASE WHEN status = 'paid' THEN 1 ELSE 0 END) / COUNT(*), 1)
        FROM orders GROUP BY user_id ORDER BY user_id""")
```

**Cost:** O(n) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two grouped queries, joined. That costs O(n) time and O(u) space."

"I can do better: `SUM(CASE ...)`. That's O(n) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/sql-foundations/share-of-paid-orders)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. WHERE vs HAVING?</b></summary>

WHERE filters rows before grouping. HAVING filters groups after grouping, so it can use aggregates.

</details>

<details>
<summary><b>2. What's the logical order of a SQL query?</b></summary>

FROM and JOIN, WHERE, GROUP BY, HAVING, SELECT, ORDER BY, LIMIT.

</details>

<details>
<summary><b>3. INNER vs LEFT JOIN?</b></summary>

INNER keeps matches only. LEFT keeps all left rows, and fills in NULL where there's no match.

</details>

<details>
<summary><b>4. Two tables of 3 rows are joined. How many rows can you get?</b></summary>

From 0 to 9, depending on matches. Duplicate keys multiply.

</details>

<details>
<summary><b>5. UNION vs UNION ALL?</b></summary>

UNION removes duplicates, which costs extra work. UNION ALL keeps everything, and is faster.

</details>

<details>
<summary><b>6. EXISTS vs IN vs JOIN?</b></summary>

EXISTS checks for any match and is safe with NULLs. IN checks a list, but NOT IN breaks on NULLs. JOIN brings in columns, and can multiply rows.

</details>

<details>
<summary><b>7. Scenario: after adding a join, revenue doubled. What happened?</b></summary>

The join key isn't unique on one side, so rows multiplied. Check row counts before and after, and check key uniqueness, then aggregate or deduplicate before joining.

</details>

<details>
<summary><b>8. Scenario: your LEFT JOIN returns the same as an INNER JOIN. Why?</b></summary>

Probably a WHERE condition on the right table is removing NULL rows. Move it into the ON clause.

</details>

<details>
<summary><b>9. Scenario: NOT IN returns zero rows, though you're sure there are matches. Why?</b></summary>

The subquery has a NULL. Use NOT EXISTS, or a LEFT JOIN with IS NULL.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Your first queries
- [ ] I can explain: Summarising: GROUP BY and HAVING
- [ ] I can explain: Joins, and why totals go wrong
- [ ] I can explain: Subqueries, EXISTS and UNION

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Subquery vs Average
- [ ] [Q17](#q17) EXISTS
- [ ] [Q18](#q18) Monthly Totals
- [ ] [Q19](#q19) Top Spender
- [ ] [Q20](#q20) Share of Paid Orders

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.